### Semantic Search

In the previous demo, we turned sentences into **embeddings** with a Sentence Transformer and compared two sentences using **cosine similarity**. We even wrote a little `find_most_similar` function that scored one sentence against a list of others.

Semantic search is that same idea, turned into something genuinely useful. Instead of comparing two sentences, we:

1. Embed a whole **corpus** (a collection of documents) once and keep those vectors around.
2. Embed an incoming **query**.
3. Return the corpus entries whose embeddings are **closest** to the query — i.e. the ones closest in *meaning*.

The important word is *meaning*. A traditional keyword search matches literal words, so "how do I get my money back?" would miss a document titled "refund policy". Semantic search embeds both into the same vector space, where *refund* and *money back* sit close together, so the right document is retrieved even with **zero shared keywords**. It also handles synonyms, paraphrases, and even typos gracefully.

This is the retrieval building block behind things like FAQ bots, document search, recommendation, and Retrieval-Augmented Generation (RAG).

We'll keep everything consistent with the previous demo: the same library (`sentence-transformers`), the same model (`all-MiniLM-L6-v2`), and the same `encode` + cosine-similarity machinery. The only new tool is `util.semantic_search`, which is essentially a fast, batched version of the `find_most_similar` helper we wrote by hand.

In [2]:
!pip install -q -U sentence-transformers

In [3]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
import pandas as pd

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

Same model as before: it maps any sentence to a 384-dimensional embedding.

In [4]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

model

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

#### The corpus

In a real system this could be thousands of FAQ answers, product descriptions, or wiki pages.

In [5]:
corpus = [
    "You can reset your password from the account settings page under 'Security'.", #0
    "Our refund policy allows returns within 30 days of purchase for a full refund.",
    "The mobile app is available for both iOS and Android devices.",
    "Standard shipping usually takes three to five business days to arrive.",
    "To cancel your subscription, open Billing and choose 'Cancel plan'.",
    "We accept all major credit cards as well as PayPal at checkout.",
    "Two-factor authentication adds an extra layer of security to your login.", #6
    "You can reach our support team by email or through the live chat widget.", #7
    "Downloaded invoices can be found in the Billing section of your dashboard.",
    "Premium members get free express shipping on every order.",
    "You can change the language of the interface from the display settings.",
    "Orders can be tracked in real time from the 'My Orders' page.",
]

len(corpus)

12

#### Embed the corpus (once)

Encoding is the expensive step, so we do it a **single time** and reuse the result for every query that comes in later. We pass `convert_to_tensor=True` because `util.semantic_search` works with PyTorch tensors.

In [6]:
corpus_embeddings = model.encode(corpus, convert_to_tensor=True, show_progress_bar=True)

corpus_embeddings.shape

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

torch.Size([12, 384])

Twelve documents, each a 384-dimensional vector — exactly the embedding size we saw in the previous demo.

#### Searching with `util.semantic_search`

For a query, we embed it the same way, then hand the query embedding and the corpus embeddings to `util.semantic_search`. By default it computes **cosine similarity** between the query and every document and returns the top matches.

It returns a list (one entry per query) of lists of hits. Each hit is a small dictionary with:
- `corpus_id`: the index of the matching document in our `corpus` list
- `score`: the cosine similarity (higher = more similar)

In [7]:
query = "I forgot my login details, how do I get back into my account?"

query_embedding = model.encode(query, convert_to_tensor=True)

hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=3)

hits

[[{'corpus_id': 0, 'score': 0.6035077571868896},
  {'corpus_id': 6, 'score': 0.302597314119339},
  {'corpus_id': 7, 'score': 0.27712953090667725}]]

Notice the top hit is the password-reset document — even though the query never says "reset" or "password".

In [8]:
def search(query, top_k=3):
    query_embedding = model.encode(query, convert_to_tensor=True)
    hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=top_k)[0]

    print(f"Query: {query}\n")
    for rank, hit in enumerate(hits, start=1):
        doc = corpus[hit["corpus_id"]]
        print(f"{rank}. (score: {hit['score']:.4f})  {doc}")

In [9]:
search("I forgot my login details, how do I get back into my account?")

Query: I forgot my login details, how do I get back into my account?

1. (score: 0.6035)  You can reset your password from the account settings page under 'Security'.
2. (score: 0.3026)  Two-factor authentication adds an extra layer of security to your login.
3. (score: 0.2771)  You can reach our support team by email or through the live chat widget.


Try a few more queries phrased in everyday language — none of them share the exact wording of the documents they should retrieve.

In [10]:
search("How long until my package shows up?")

Query: How long until my package shows up?

1. (score: 0.5822)  Standard shipping usually takes three to five business days to arrive.
2. (score: 0.2781)  Orders can be tracked in real time from the 'My Orders' page.
3. (score: 0.2125)  Our refund policy allows returns within 30 days of purchase for a full refund.


In [11]:
search("Can I get my money back if I change my mind?")

Query: Can I get my money back if I change my mind?

1. (score: 0.3199)  Our refund policy allows returns within 30 days of purchase for a full refund.
2. (score: 0.2857)  You can reset your password from the account settings page under 'Security'.
3. (score: 0.2556)  To cancel your subscription, open Billing and choose 'Cancel plan'.


In [12]:
search("Is there a version for my phone?")

Query: Is there a version for my phone?

1. (score: 0.4075)  The mobile app is available for both iOS and Android devices.
2. (score: 0.1846)  You can reach our support team by email or through the live chat widget.
3. (score: 0.1469)  To cancel your subscription, open Billing and choose 'Cancel plan'.


#### Semantic vs. keyword search

To make the point concrete, let's write a naive **keyword** search that scores documents by how many query words they literally contain, and compare it head-to-head with our semantic search on a query where the wording doesn't line up.

In [13]:
def keyword_search(query, top_k=3):
    query_words = set(query.lower().split())
    scored = []

    for doc in corpus:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words)   # count of shared words
        scored.append((overlap, doc))

    scored.sort(key=lambda x: x[0], reverse=True)

    print(f"Query: {query}\n")
    for overlap, doc in scored[:top_k]:
        print(f"- (shared words: {overlap})  {doc}")

In [14]:
tricky_query = "Can I get my money back if I change my mind?"

print("=== KEYWORD SEARCH ===")
keyword_search(tricky_query)

print("\n=== SEMANTIC SEARCH ===")
search(tricky_query)

=== KEYWORD SEARCH ===
Query: Can I get my money back if I change my mind?

- (shared words: 2)  You can change the language of the interface from the display settings.
- (shared words: 1)  You can reset your password from the account settings page under 'Security'.
- (shared words: 1)  You can reach our support team by email or through the live chat widget.

=== SEMANTIC SEARCH ===
Query: Can I get my money back if I change my mind?

1. (score: 0.3199)  Our refund policy allows returns within 30 days of purchase for a full refund.
2. (score: 0.2857)  You can reset your password from the account settings page under 'Security'.
3. (score: 0.2556)  To cancel your subscription, open Billing and choose 'Cancel plan'.


The keyword search flails: the words *money* and *back* never appear in the refund document, so it can't connect them. Semantic search ranks the refund policy first, because it compares **meaning**, not spelling. That single difference is the whole reason semantic search exists.

#### Scores and thresholds — knowing when there's *no* good answer

The scores are cosine similarities, so they live between -1 and 1 (for these models, relevant matches typically land somewhere around 0.3–0.7). Because `top_k` always returns *something*, it's good practice to set a **threshold** and treat weak top scores as "no confident match" — otherwise a search for something totally unrelated will still return your closest-but-irrelevant document.

In [15]:
def search_with_threshold(query, top_k=3, threshold=0.35):
    query_embedding = model.encode(query, convert_to_tensor=True)
    hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=top_k)[0]

    print(f"Query: {query}")
    if hits[0]["score"] < threshold:
        print(f"  No confident match (best score {hits[0]['score']:.4f} < {threshold})\n")
        return

    for rank, hit in enumerate(hits, start=1):
        print(f"  {rank}. (score: {hit['score']:.4f})  {corpus[hit['corpus_id']]}")
    print()

In [16]:
search_with_threshold("How do I turn on MFA?")
search_with_threshold("What is the capital of France?")

Query: How do I turn on MFA?
  1. (score: 0.3720)  Two-factor authentication adds an extra layer of security to your login.
  2. (score: 0.3507)  You can reset your password from the account settings page under 'Security'.
  3. (score: 0.3114)  You can reach our support team by email or through the live chat widget.

Query: What is the capital of France?
  No confident match (best score 0.1014 < 0.35)

